# 02 — KL Divergence

Measure the cost of a wrong model — the *extra bits* you pay for coding with $q$ when the truth is $p$. Build the formula from the mismatched code of `02/01`, prove by hand that the cost is never negative (Gibbs' inequality) and collect what that one fact buys, then learn the three ways relative entropy is *not* a distance — asymmetric, infinite off-support, no triangle inequality — and the three ways it behaves better than one: additive, convex, and monotone under any processing. This is the parent quantity of the entropic side of the course — mutual information, transfer entropy, Sinkhorn and the Umegaki lift are all relative entropies in disguise (the Bures distance of `02/12` and the Wasserstein distances of module 03 are the *other* family).

**Prerequisites:** `02/01_shannon_entropy` (surprise, $H(p)$, the source-coding floor and the cost of a mismatched code — today made exact), `01/16_tensor_and_partial_trace` (joint tables, products, stochastic maps on probability vectors), `01/14_trace_distance_and_fidelity` (total variation — a distance; today's object is *not* one).

**What you'll be able to do**
- Define the relative entropy $D(p\|q) = \sum_x p_x\log_2\frac{p_x}{q_x}$ as *cross-entropy minus entropy*, compute it by hand, and read it as the extra bits per symbol of the wrong code — the $0.25$ bits of `02/01`'s mismatched codes, exactly.
- Prove Gibbs' inequality $D(p\|q) \ge 0$ with equality iff $p = q$ from the single inequality $\ln x \le x - 1$, and use it to re-prove $H(p) \le \log_2 n$ and the "$\ge$" of the source-coding theorem — the two promises of `02/01`.
- Show that $D$ is not a distance (asymmetric, $+\infty$ on a missing outcome, no triangle inequality) and say why that asymmetry is a feature you will *choose* from in `03/11` and `04/08`.
- Derive additivity on product distributions and the chain rule, check joint convexity, and prove monotonicity under any stochastic map (the data-processing inequality) — the property the quantum lift of `02/09` must keep.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.classical import kl_divergence, shannon_entropy

rng = np.random.default_rng(0)   # random distributions and stochastic maps for the checks of sections 2 and 4
viz.use_course_style()

## 1. Extra bits: relative entropy from the mismatched code

In `02/01` §4 you built a prefix code for the dyadic source $p = (\tfrac12, \tfrac14, \tfrac18, \tfrac18)$ — codeword lengths $\ell_x = -\log_2 p_x$, average $H(p) = 1.75$ bits, the floor — and then used it on the *wrong* source: the uniform $u = (\tfrac14,\tfrac14,\tfrac14,\tfrac14)$ cost $2.25$ bits per symbol against $H(u) = 2$, and a $2$-bit code (the optimal code for $u$) on the dyadic source cost $2$ bits against $H(p) = 1.75$. Let's make that overpayment a formula.

A code built for a model $q$ spends $-\log_2 q_x$ bits on outcome $x$ (Shannon's lengths, `02/01`). If the data actually follow $p$, the average spend is the **cross-entropy**

$$
H(p, q) = -\sum_x p_x \log_2 q_x ,
$$

and the least you could have spent is $H(p) = -\sum_x p_x\log_2 p_x$. The difference is the **relative entropy**, or Kullback–Leibler divergence (Kullback & Leibler 1951),

$$
D(p\|q) = H(p, q) - H(p) = \sum_x p_x \log_2\frac{p_x}{q_x} ,
$$

the *extra bits per symbol* of believing $q$ when the truth is $p$ — "$p$ relative to $q$", truth first, model second. Term by term it is the average, under the truth, of the **surprise gap** $\log_2\frac{p_x}{q_x} = (-\log_2 q_x) - (-\log_2 p_x)$: how much more surprised the model is than it should be. Conventions: $0\log\frac{0}{q} = 0$ (an outcome that never happens costs nothing), and $p\log\frac{p}{0} = +\infty$ for $p > 0$ — we return to that in section 3. Units: $\log_2$ gives bits, as in `02/01`; the helper's `base` argument switches to nats ($\ln$), which is what the entropic transport of `03/09`–`03/11` uses — there the second argument is moreover an *unnormalised* kernel, and the formula gains the terms $-\sum p + \sum q$ that vanish for probability vectors (the generalised KL; `03/11` writes the projection with $K$ unnormalised and those terms absorbed into a constant).

By hand, the two mismatches of `02/01`. Dyadic code on the uniform source: $D(u\|p) = \sum_x \tfrac14\log_2\frac{1/4}{p_x} = \tfrac14\big(\log_2\tfrac12 + \log_2 1 + \log_2 2 + \log_2 2\big) = \tfrac14(-1 + 0 + 1 + 1) = \tfrac14$ bit — the $2.25 - 2$. Two-bit code on the dyadic source: $D(p\|u) = \sum_x p_x\log_2(4p_x) = \tfrac12\cdot 1 + \tfrac14\cdot 0 + \tfrac18\cdot(-1) + \tfrac18\cdot(-1) = \tfrac14$ bit — the $2 - 1.75$. Both equal $\tfrac14$ here; that is a coincidence of this pair, and section 3 shows the two directions differ in general. Let's compute the surprise gaps term by term, the cross-entropy, and compare with the course helper `kl_divergence`.


In [ ]:
def kl_by_hand(p, q):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    if np.any((p > 0) & (q == 0)):              # the model gives zero to something that happens
        return np.inf
    nz = p > 0                                  # 0 log(0/q) = 0
    return float((p[nz] * np.log2(p[nz] / q[nz])).sum()) + 0.0

def cross_entropy(p, q):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    nz = p > 0
    return float(-(p[nz] * np.log2(q[nz])).sum())

p_dyadic = np.array([0.5, 0.25, 0.125, 0.125])
u4 = np.full(4, 0.25)
print("02/01's mismatches, made exact")
print(f"  dyadic code on the uniform source: cost H(u,p) = {cross_entropy(u4, p_dyadic):.3f}  =  H(u) {shannon_entropy(u4):.3f}  +  D(u||p) {kl_by_hand(u4, p_dyadic):.3f}")
print(f"  2-bit code on the dyadic source:   cost H(p,u) = {cross_entropy(p_dyadic, u4):.3f}  =  H(p) {shannon_entropy(p_dyadic):.3f}  +  D(p||u) {kl_by_hand(p_dyadic, u4):.3f}")

p = np.array([0.7, 0.2, 0.1])
q = np.array([0.4, 0.4, 0.2])
gaps = np.log2(p / q)
print("\nA three-outcome pair, term by term")
print("  p                 :", p)
print("  q                 :", q)
print("  surprise gap log2(p/q):", np.round(gaps, 4), "  (positive where the model under-rates the outcome)")
print("  weighted p*log2(p/q)  :", np.round(p * gaps, 4), "  -> sum =", round(float((p * gaps).sum()), 4), "bits")
print(f"  cross-entropy H(p,q) = {cross_entropy(p, q):.4f},  H(p) = {shannon_entropy(p):.4f},  difference = {cross_entropy(p, q) - shannon_entropy(p):.4f}")
print(f"  by hand D(p||q) = {kl_by_hand(p, q):.4f}   helper kl_divergence = {kl_divergence(p, q):.4f}   D(p||p) = {kl_by_hand(p, p):.4f}")
print(f"  in nats: helper base=e gives {kl_divergence(p, q, base=np.e):.4f}  =  bits x ln 2 = {kl_by_hand(p, q) * np.log(2):.4f}   (the unit of 03/09-03/11)")

**Read the output.** The two overpayments of `02/01` are exactly $\tfrac14$ bit each, and each is a cross-entropy minus an entropy: $2.25 = 2 + 0.25$ and $2 = 1.75 + 0.25$. On the three-outcome pair the surprise gap is positive on the first outcome ($p$ says $0.7$, the model only $0.4$: the model is surprised $0.81$ bits more than it should be) and negative on the other two (the model *over*-rates them), and the weighted sum lands at $0.2651$ bits — the by-hand value, the helper's value, and the cross-entropy minus entropy all agree. $D(p\|p) = 0$: the right model wastes nothing; and in nats the same divergence is $\ln 2$ times smaller, the unit entropic transport will use. The sum has negative terms, yet came out positive; the next section shows it always does.


## 2. Gibbs' inequality, and what one inequality buys

**Claim.** $D(p\|q) \ge 0$, with equality if and only if $p = q$.

**Proof by hand.** Everything rests on one picture: $\ln t \le t - 1$ for all $t > 0$, with equality only at $t = 1$ (the tangent line at $1$ sits above the concave $\ln$). Apply it to $t = q_x/p_x$ on the support of $p$:

$$
-D(p\|q)\,\ln 2 \;=\; \sum_x p_x \ln\frac{q_x}{p_x} \;\le\; \sum_x p_x\Big(\frac{q_x}{p_x} - 1\Big) \;=\; \sum_{x:\,p_x>0} q_x - 1 \;\le\; 0 .
$$

The first "$\le$" is an equality only if $q_x/p_x = 1$ wherever $p_x > 0$; the second only if $q$ puts no mass outside the support of $p$. Together: equality iff $p = q$. That is the whole proof (Cover & Thomas, Thm 2.6.3).

Two promises of `02/01` are now due, and both are one line each.

*The uniform is the most uncertain — second proof.* Take $q = u = (\tfrac1n, \dots, \tfrac1n)$: $D(p\|u) = \sum_x p_x\log_2(n\,p_x) = \log_2 n - H(p) \ge 0$, so $H(p) \le \log_2 n$ with equality iff $p = u$ — Jensen was `02/01`'s route; Gibbs is the shorter one, and it says *what* the gap is: $\log_2 n - H(p)$ is the relative entropy from $p$ to the uniform.

*The "$\ge$" of the source-coding theorem.* Take any prefix code with lengths $\ell_x$ and Kraft sum $Z = \sum_x 2^{-\ell_x} \le 1$ (Kraft's inequality, quoted in `02/01` — its proof is a counting argument on the binary tree; today's proof is conditional on it). Define the model the code implicitly believes, $q_x = 2^{-\ell_x}/Z$, a probability vector. Then $\ell_x = -\log_2 q_x - \log_2 Z$ and the average length is
$$
\bar\ell = \sum_x p_x\ell_x = H(p, q) - \log_2 Z = H(p) + D(p\|q) - \log_2 Z \;\ge\; H(p),
$$
since $D \ge 0$ and $-\log_2 Z \ge 0$. Equality needs $D = 0$ (lengths exactly the surprises) and $Z = 1$ (no slack) — exactly the dyadic code. Every code pays its relative entropy to the source. Let's draw the tangent-line picture, then check both consequences numerically on random distributions and random codes.


In [ ]:
# the one inequality: ln x <= x - 1
xs = np.linspace(0.05, 3.0, 300)
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(xs, np.log(xs), color=viz.SOURCE_COLOR, lw=2.2, label="ln x")
ax.plot(xs, xs - 1, color=viz.TARGET_COLOR, lw=2.2, label="x - 1  (tangent at x = 1)")
ax.fill_between(xs, np.log(xs), xs - 1, color=COLORS["highlight"], alpha=0.25, label="the gap: x - 1 - ln x >= 0")
ax.axvline(1.0, color=COLORS["muted"], lw=1, ls="--"); ax.axhline(0.0, color=COLORS["muted"], lw=1, ls="--")
ax.scatter([1.0], [0.0], color=COLORS["text"], zorder=5)
ax.set_xlabel("x"); ax.set_ylabel("value")
ax.set_title("Gibbs' inequality rests on one picture: ln x touches x - 1 only at x = 1", pad=12)
ax.legend(loc="lower right")
plt.show()

# consequence 1: D(p || uniform) = log2 n - H(p) >= 0, equality only at the uniform
n = 5
u = np.full(n, 1.0 / n)
ps = rng.dirichlet(np.ones(n), size=2000)
gap = np.array([kl_by_hand(pi, u) for pi in ps])
print(f"2000 random 5-outcome p:  max |D(p||u) - (log2 5 - H(p))| = {np.max(np.abs(gap - (np.log2(n) - np.array([shannon_entropy(pi) for pi in ps])))):.2e}")
print(f"                          min D(p||u) = {gap.min():.4f} bits (>= 0), D(u||u) = {kl_by_hand(u, u):.4f}")

# consequence 2: every prefix code pays D to the source.  Random Kraft-feasible lengths -> implicit model q
p5 = rng.dirichlet(np.ones(n))
worst = np.inf
for _ in range(2000):
    lengths = rng.integers(1, 8, size=n)
    Z = (2.0 ** -lengths).sum()
    if Z > 1:                      # not a prefix code (Kraft fails): skip
        continue
    q_code = 2.0 ** -lengths / Z
    avg_len = (p5 * lengths).sum()
    identity = shannon_entropy(p5) + kl_by_hand(p5, q_code) - np.log2(Z)
    assert abs(avg_len - identity) < 1e-12
    worst = min(worst, avg_len - shannon_entropy(p5))
print(f"random prefix codes on a random source: average length - H(p) >= {worst:.4f} bits in every case (the code's D(p||q) - log2 Z)")

**Read the figure and the output.** The logarithm lies below its tangent at $1$ everywhere and touches it only there — that one rose-shaded gap is Gibbs' inequality; the sum of the gaps, weighted by $p$, is $D(p\|q)\ln 2$. Two thousand random five-outcome distributions all satisfy $D(p\|u) = \log_2 5 - H(p)$ to machine precision and none goes below zero; the uniform itself gives exactly $0$. And for every Kraft-feasible code tried on a random source, the average length is exactly $H(p) + D(p\|q) - \log_2 Z$ — the identity holds to $10^{-12}$ — so the excess over the entropy is never negative: a code can only match the floor by believing the truth, with no slack. One inequality, three theorems.


## 3. Three ways it is *not* a distance

The total variation of `01/14` is a distance: symmetric, zero only at equality, triangle inequality. Relative entropy keeps only the middle property. That is not a defect — it is what a *coding cost* looks like — but you must know the three failures by heart, because each one reappears in the quantum lift (`02/09`) and in the transport problems of modules 03 and 04.

**(i) Asymmetry.** $D(p\|q)$ is the cost of believing $q$ when $p$ is true; $D(q\|p)$ is the opposite mistake. Nothing forces them equal, and they rarely are. Fix $p$ and let $q$ range over the whole simplex of three outcomes: the two landscapes $q \mapsto D(p\|q)$ and $q \mapsto D(q\|p)$ are different surfaces that share one zero, at $q = p$.

**(ii) Infinity off the support.** If $q_x = 0$ for an outcome with $p_x > 0$, then $D(p\|q) = +\infty$: the model has *no codeword* for an event that happens, and no finite number of bits can describe it. The reverse — $p_x = 0$ where $q_x > 0$ — costs nothing: $0\log\frac{0}{q} = 0$, a codeword you never use. So $D(p\|q) < \infty$ requires $\mathrm{supp}\,p \subseteq \mathrm{supp}\,q$. In the landscape, the edges of the simplex where $q$ has a zero are walls of infinite height for $D(p\|q)$; for $D(q\|p)$ (with $p$ strictly positive) the edges are finite and the corners are the three costs $\log_2\frac1{p_x}$.

**(iii) No triangle inequality.** There are triples with $D(p\|r) > D(p\|q) + D(q\|r)$ — going through a midpoint can be cheaper than going direct. Let's draw the two landscapes and exhibit all three failures in numbers. (The asymmetry is why you will *choose* a direction: Sinkhorn (`03/10`), read as a KL projection in `03/11`, minimises $D(\pi\|K)$ — plan first, kernel second — and that choice decides what the solution looks like.)


In [ ]:
def kl_landscapes(p, m=300):
    # q = (q1, q2, 1 - q1 - q2) on a grid over the simplex; both directions of D
    g = np.linspace(0.0, 1.0, m)
    Q1, Q2 = np.meshgrid(g, g)
    Q3 = 1.0 - Q1 - Q2
    inside = Q3 >= 0
    fwd = np.full(Q1.shape, np.nan); bwd = np.full(Q1.shape, np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        Q = np.stack([Q1, Q2, Q3], axis=-1)
        fwd[inside] = np.sum(p * np.log2(p / Q[inside]), axis=-1)                 # D(p || q): +inf on the edges
        Qi = Q[inside]
        terms = np.where(Qi > 0, Qi * np.log2(Qi / p), 0.0)                       # 0 log 0 = 0
        bwd[inside] = terms.sum(axis=-1)                                          # D(q || p): finite, p > 0
    return Q1, Q2, fwd, bwd

Q1, Q2, fwd, bwd = kl_landscapes(p)
vmax = 2.0
fig, axes = plt.subplots(1, 3, figsize=(16, 5.0), gridspec_kw={"width_ratios": [1, 1, 0.95], "wspace": 0.35})
for ax, Z, title in [(axes[0], fwd, "D(p || q): model q varies"), (axes[1], bwd, "D(q || p): truth q varies")]:
    im = ax.contourf(Q1, Q2, np.clip(Z, 0, vmax), levels=np.linspace(0, vmax, 21), cmap=viz.CMAP_COST)
    ax.contour(Q1, Q2, np.clip(Z, 0, vmax), levels=[0.1, 0.25, 0.5, 1.0], colors=[COLORS["text"]], linewidths=0.7)
    ax.plot([0, 1, 0, 0], [0, 0, 1, 0], color=COLORS["muted"], lw=1)
    ax.scatter([p[0]], [p[1]], color=COLORS["highlight"], s=60, zorder=5, label="q = p  (the only zero)")
    ax.set_xlabel("q1"); ax.set_ylabel("q2"); ax.set_title(title, pad=10); ax.set_aspect("equal")
    ax.legend(loc="upper right")
fig.colorbar(im, ax=axes[:2], shrink=0.85, label="bits (clipped at 2)")
# third panel: walk from p straight into the q3 = 0 edge, both directions
q_hole = np.array([0.5, 0.5, 0.0])
q3_walk = np.logspace(-1, -8, 300)                    # third entry of q_t, from p's 0.1 down towards 0
walk = [(1 - t) * p + t * q_hole for t in 1 - q3_walk / p[2]]   # q_t = (1-t) p + t q_hole has q3 = (1-t) p3
ax = axes[2]
ax.plot(q3_walk, [kl_by_hand(p, qt) for qt in walk], color=viz.SOURCE_COLOR, lw=2.4, label="D(p || q_t): diverges like 0.1 log2(0.1/q3)")
ax.plot(q3_walk, [kl_by_hand(qt, p) for qt in walk], color=viz.TARGET_COLOR, lw=2.4, label="D(q_t || p): finite, -> 0.418")
ax.axhline(kl_by_hand(q_hole, p), color=viz.TARGET_COLOR, lw=1, ls="--")
ax.set_xscale("log"); ax.invert_xaxis()
ax.set_xlabel("third entry q3 of q_t  (log axis, towards the edge)"); ax.set_ylabel("bits")
ax.set_title("Walking into the edge q3 = 0", pad=10); ax.legend(loc="upper left")
for a_ in axes[:2]:
    a_.plot([p[0], q_hole[0]], [p[1], q_hole[1]], color=COLORS["text"], lw=1, ls=":")
fig.suptitle("Two landscapes, one shared zero: relative entropy is asymmetric, and infinite off the support", y=1.04)
plt.show()

print(f"(i)  asymmetry:  D(p||q) = {kl_by_hand(p, q):.4f}   D(q||p) = {kl_by_hand(q, p):.4f}   (p = {p}, q = {q})")
q_hole = np.array([0.5, 0.5, 0.0])
print(f"(ii) support:    D(p||q_hole) = {kl_by_hand(p, q_hole)}   (q_hole = {q_hole} has no codeword for outcome 3, which p uses 10% of the time)")
print(f"                 D(q_hole||p) = {kl_by_hand(q_hole, p):.4f}   (the unused codeword costs nothing)")
print(f"                 corner costs of the right panel: log2(1/p) = {np.round(np.log2(1 / p), 4)} bits at the three vertices")
r = np.array([0.1, 0.2, 0.7])
print(f"(iii) triangle:  D(p||r) = {kl_by_hand(p, r):.4f}  >  D(p||q) + D(q||r) = {kl_by_hand(p, q):.4f} + {kl_by_hand(q, r):.4f} = {kl_by_hand(p, q) + kl_by_hand(q, r):.4f}")

**Read the figure and the output.** The two landscapes are different surfaces — same fixed $p = (0.7, 0.2, 0.1)$, same single zero at the rose point $q = p$, different contours everywhere else. On the left ($D(p\|q)$, the model varies) the wall is plain along the $q_1 = 0$ edge, where the model denies the outcome $p$ uses $70\%$ of the time; along the other two edges the wall is there too but thin — $p_2 = 0.2$ and $p_3 = 0.1$ only push $p_x\log_2(p_x/q_x)$ past the colour scale in a sliver narrower than the grid — so the third panel walks straight at the $q_3 = 0$ edge along the segment $q_t = (1-t)\,p + t\,q_{\text{hole}}$, $q_{\text{hole}} = (0.5, 0.5, 0)$, and there the two directions part company: $D(p\|q_t)$ climbs without bound as $q_t$ loses its third entry, while $D(q_t\|p)$ ends at the finite $0.418$ bits of $q_{\text{hole}}$ (the walk is drawn against the vanishing third entry $q_3$ on a log axis: the divergence is logarithmic, $0.1\log_2(0.1/q_3)$, slow but without bound). `kl_by_hand` returns `inf` at the endpoint, as it must (so does the helper). On the right ($D(q\|p)$, the truth varies) the edges are harmless everywhere — a truth that never uses an outcome wastes nothing — and the three corners are exactly $\log_2\frac1{p_x}$: a truth concentrated on one outcome pays the surprise of that outcome under $p$. The numbers give the rest: $0.2651 \ne 0.2771$ bits (asymmetry), and the detour $p \to q \to r$ costs $1.10$ bits where the direct $p \to r$ costs $1.68$ (no triangle inequality). A coding cost, not a ruler. The geometry that *does* make the simplex a metric space — Fisher–Rao — is `02/06`; and relative entropy is not unrelated to it: the Hessian of $q \mapsto D(p\|q)$ at $q = p$ is the Fisher information matrix, which is why the two landscapes look alike *near* the zero (both are quadratic bowls there) and differ only far from it.


## 4. Three ways it behaves *better* than a distance

**(a) Additivity on products.** For independent pairs, $p = p_A \otimes p_B$ and $q = q_A \otimes q_B$ (the product tables of `01/16`), the log of a product is a sum:
$$
D(p_A\!\otimes\! p_B\,\|\,q_A\!\otimes\! q_B) = \sum_{a,b} p_A(a)p_B(b)\Big[\log_2\tfrac{p_A(a)}{q_A(a)} + \log_2\tfrac{p_B(b)}{q_B(b)}\Big] = D(p_A\|q_A) + D(p_B\|q_B),
$$
because each bracket's partner sums to one. Extra bits add over independent symbols — as lengths of codewords do. More generally, for any joint tables, the **chain rule** $D(p_{AB}\|q_{AB}) = D(p_A\|q_A) + \sum_a p_A(a)\,D(p_{B|a}\|q_{B|a})$: first the marginals, then the conditionals averaged over the truth (same expansion, $p_{AB} = p_A\,p_{B|A}$). The chain rule of entropy in `02/01` §3 is this one with $q$ uniform.

**(b) Joint convexity.** $D(\lambda p_1 + (1-\lambda)p_2\,\|\,\lambda q_1 + (1-\lambda)q_2) \le \lambda D(p_1\|q_1) + (1-\lambda)D(p_2\|q_2)$: mixing two sources and two models costs no more than the mixture of the costs. The engine is the **log-sum inequality**, and it is Jensen (`02/01` §2) once more. For non-negative $a_i, b_i$ with $B = \sum_i b_i$, let $f(t) = t\log t$ (convex: $f'' = 1/t > 0$) and write
$$
\sum_i a_i\log\frac{a_i}{b_i} = B\sum_i \frac{b_i}{B}\,f\Big(\frac{a_i}{b_i}\Big) \;\ge\; B\,f\Big(\sum_i \frac{b_i}{B}\frac{a_i}{b_i}\Big) = \Big(\sum_i a_i\Big)\log\frac{\sum_i a_i}{\sum_i b_i},
$$
with equality iff all the ratios $a_i/b_i$ are equal (Cover & Thomas Thm 2.7.1). Joint convexity is this inequality applied outcome by outcome: for each $x$ take the two terms $a_1 = \lambda p_1(x), a_2 = (1-\lambda)p_2(x)$, $b_1 = \lambda q_1(x), b_2 = (1-\lambda)q_2(x)$ — the log-sum bound says $\lambda p_1(x)\log\frac{p_1(x)}{q_1(x)} + (1-\lambda)p_2(x)\log\frac{p_2(x)}{q_2(x)} \ge \big(\lambda p_1(x) + (1-\lambda)p_2(x)\big)\log\frac{\lambda p_1(x) + (1-\lambda)p_2(x)}{\lambda q_1(x) + (1-\lambda)q_2(x)}$ — and sum over $x$. In particular $D(\cdot\|q)$ is *strictly* convex in its first argument (strict convexity of $t\log t$), which is what makes "minimise $D(\pi\|K)$ under linear constraints" a problem with a unique answer: the KL projections of `03/11` and `04/08`.

**(c) Monotonicity under processing — the data-processing inequality.** Push both distributions through the same stochastic map $T$ (a Markov kernel, the lossy classical evolution of `01/19`'s dictionary row: columns are probability vectors): $D(Tp\|Tq) \le D(p\|q)$. Proof by hand, from the log-sum inequality applied to each output $y$ with $a_x = T_{yx}p_x$, $b_x = T_{yx}q_x$:
$$
D(Tp\|Tq) = \sum_y \Big(\sum_x T_{yx}p_x\Big)\log_2\frac{\sum_x T_{yx}p_x}{\sum_x T_{yx}q_x}
\;\le\; \sum_y\sum_x T_{yx}\,p_x\log_2\frac{p_x}{q_x} = \sum_x p_x\log_2\frac{p_x}{q_x} = D(p\|q),
$$
using $\sum_y T_{yx} = 1$. Merging two outcomes into one is the simplest $T$ (coarse-graining): a coarser description can only make two sources look *more* alike. No processing, deterministic or random, can manufacture distinguishability — the same statement the trace distance obeys under channels (an exercise of `01/19`), and the property a quantum relative entropy *must* keep to deserve the name — Lindblad proved it for Umegaki's in 1975 (Uhlmann 1977 in general). Let's check all three on random tables and random maps.


In [ ]:
# (a) additivity on products, and the chain rule on a correlated table
pA, qA = rng.dirichlet(np.ones(3)), rng.dirichlet(np.ones(3))
pB, qB = rng.dirichlet(np.ones(2)), rng.dirichlet(np.ones(2))
print(f"(a) product tables: D(pA x pB || qA x qB) = {kl_by_hand(np.outer(pA, pB).ravel(), np.outer(qA, qB).ravel()):.6f}  =  D(pA||qA) + D(pB||qB) = {kl_by_hand(pA, qA) + kl_by_hand(pB, qB):.6f}")
PAB = rng.dirichlet(np.ones(6)).reshape(3, 2); QAB = rng.dirichlet(np.ones(6)).reshape(3, 2)
pA_, qA_ = PAB.sum(1), QAB.sum(1)
chain = kl_by_hand(pA_, qA_) + sum(pA_[a] * kl_by_hand(PAB[a] / pA_[a], QAB[a] / qA_[a]) for a in range(3))
print(f"    chain rule on a correlated 3x2 table: D(P||Q) = {kl_by_hand(PAB.ravel(), QAB.ravel()):.6f}  =  marginal term + averaged conditionals = {chain:.6f}")

# (b) joint convexity on 2000 random quadruples
viol = -np.inf
for _ in range(2000):
    p1, p2, q1, q2 = rng.dirichlet(np.ones(4), size=4)
    lam = rng.uniform()
    lhs = kl_by_hand(lam * p1 + (1 - lam) * p2, lam * q1 + (1 - lam) * q2)
    rhs = lam * kl_by_hand(p1, q1) + (1 - lam) * kl_by_hand(p2, q2)
    viol = max(viol, lhs - rhs)
print(f"(b) joint convexity: max over 2000 random mixtures of  D(mix || mix) - mixture of D  = {viol:.3e}  (never positive)")

# (c) data processing: random stochastic maps T (columns sum to 1), and the simplest one, merging two bins
worst = -np.inf
for _ in range(2000):
    T = rng.dirichlet(np.ones(3), size=3).T           # 3x3, each column a probability vector
    worst = max(worst, kl_by_hand(T @ p, T @ q) - kl_by_hand(p, q))
T_merge12 = np.array([[1, 1, 0], [0, 0, 1]], dtype=float)   # outcomes 1 and 2 become one
T_merge23 = np.array([[1, 0, 0], [0, 1, 1]], dtype=float)   # outcomes 2 and 3 become one
print(f"(c) data processing: max over 2000 random stochastic T of  D(Tp||Tq) - D(p||q)  = {worst:.3e}  (never positive)")
print(f"    merging outcomes 1 and 2:  D(p||q) = {kl_by_hand(p, q):.4f}  ->  D(Tp||Tq) = {kl_by_hand(T_merge12 @ p, T_merge12 @ q):.4f}   (p -> {T_merge12 @ p}, q -> {T_merge12 @ q}; ratios p/q were {p[0]/q[0]:.2f} and {p[1]/q[1]:.2f})")
print(f"    merging outcomes 2 and 3:  D(p||q) = {kl_by_hand(p, q):.4f}  ->  D(Tp||Tq) = {kl_by_hand(T_merge23 @ p, T_merge23 @ q):.4f}   (p -> {T_merge23 @ p}, q -> {T_merge23 @ q}; ratios p/q were {p[1]/q[1]:.2f} and {p[2]/q[2]:.2f} -- equal)")
T_forget = np.full((3, 3), 1 / 3)                       # the map that forgets everything
print(f"    forgetting everything (T = uniform columns): D(Tp||Tq) = {kl_by_hand(T_forget @ p, T_forget @ q):.4f}")

**Read the output.** Additivity holds to six decimals on a random product pair, and the chain rule to six decimals on a random correlated table: relative entropy decomposes exactly the way codeword lengths do — marginal first, then the conditionals weighted by the truth. Over two thousand random mixtures, the left side of joint convexity never exceeds the right (the largest value of left minus right is $-6\times10^{-4}$ — still negative). And over two thousand random stochastic maps, $D(Tp\|Tq)$ never exceeds $D(p\|q)$. The two merges are the lesson in miniature: merging outcomes $1$ and $2$ drops $0.265$ bits to $0.053$ — the model's two mistakes ($0.7$ vs $0.4$ and $0.2$ vs $0.4$) partly cancel once the outcomes are indistinguishable — while merging outcomes $2$ and $3$ loses *nothing*, $0.265 \to 0.265$, because there $p/q$ was $0.5$ on both: the log-sum inequality is an equality exactly when the ratios agree, and a coarse-graining that keeps the ratios keeps the whole cost. The map that forgets everything sends both distributions to the same point, $D = 0$. Distinguishability is a resource that processing can only spend.


## 5. Where the parent quantity goes

Three objects in this course are relative entropies in disguise, and you can now name each.

- **Mutual information** (`02/03`): how far is a joint table from the product of its marginals? $I(A{:}B) = D(p_{AB}\,\|\,p_A \otimes p_B)$. Gibbs gives $I \ge 0$ with equality iff independent; by the chain rule of 4(a) it is also $\sum_a p_A(a)\,D(p_{B|a}\|p_B)$ — how much, on average, knowing $a$ moves the distribution of $b$. (On the correlated coins of `01/16` it is $1$ bit; on the independent table, $0$ — `02/03` computes it.)
- **Entropic optimal transport** (`03/09`–`03/11`): Sinkhorn's plan is the table $\pi$ with prescribed marginals that minimises $D(\pi\,\|\,K)$ to a Gibbs kernel $K = e^{-C/\varepsilon}$ — a **KL projection**. Strict convexity in the first argument (4b) makes it unique; the chosen direction (plan first) is what keeps the plan inside the support of $K$ (3ii).
- **Umegaki relative entropy** (`02/09`): replace $p, q$ by density matrices, $S(\rho\|\sigma) = \mathrm{tr}\,\rho(\log_2\rho - \log_2\sigma)$. When $\rho$ and $\sigma$ are diagonal in the same basis it *is* $D$ of the diagonals; it inherits $\ge 0$ (Klein), $+\infty$ when the support of $\rho$ leaves that of $\sigma$, and monotonicity under every channel (Lindblad 1975; Uhlmann 1977) — 4(c), lifted. What it adds is the corners: $|{+}\rangle$ and $I/2$ share the diagonal $(\tfrac12, \tfrac12)$, so the classical $D$ of their Born probabilities is $0$, while $S(|{+}\rangle\langle{+}|\,\|\,I/2) = 1$ bit — `01/13`'s seed, finally seen by a relative entropy.

Two small remarks before you go. Relative entropy is invariant under *relabelling* the outcomes (the sum does not care about the order of its terms) — which is exactly why `02/08` will call it a *vertical* comparison, blind to any geometry of the bins; and you have already applied it to *tables*: the chain-rule check of section 4 fed $3\times2$ joint tables to `kl_by_hand` by flattening them with `.ravel()`, which is all `02/03` needs.

Today you built the classical column of that last row. Everything else is a lift.


## Your turn

1. **Both ways, and a detour.** Pick your own $p, q$ on three outcomes and confirm $D(p\|q) \ne D(q\|p)$. Then search (a few hundred random triples) for the triple $p, q, r$ where $D(p\|q) + D(q\|r) - D(p\|r)$ is most negative — the worst triangle-inequality failure you can find — and describe the three distributions: where does $q$ sit relative to $p$ and $r$?
2. **The hole, and how practitioners fill it.** With `p = [0.7, 0.2, 0.1]` and `q = [0.5, 0.5, 0.0]`, $D(p\|q) = \infty$. Smooth the model, $q_\epsilon = (1 - \epsilon)q + \epsilon u$, and plot $D(p\|q_\epsilon)$ against $\epsilon$ on a log axis from $10^{-6}$ to $1$: how fast does the cost fall as the model admits the missing outcome, and what is the price at $\epsilon = 1$ (the model that admits everything equally)? This is why estimated distributions are smoothed before a KL is taken — and why the Gibbs kernel $K = e^{-C/\varepsilon}$ of `03/09` is strictly positive by construction.
3. **The binary relative entropy.** For two coins, $d(p\|q) = p\log_2\frac pq + (1-p)\log_2\frac{1-p}{1-q}$. Fix $q = 0.3$ and plot $d(p\|q)$ and $d(q\|p)$ for $p \in (0, 1)$ on the same axes; mark where each blows up, and check $d(p\|\tfrac12) = 1 - h(p)$ from `02/01`. Then show numerically that $d(p\|q) \ge \frac{2}{\ln 2}(p - q)^2$ and find where the ratio $d(p\|q)\big/\frac{2}{\ln 2}(p-q)^2$ is smallest over $p \ne q$.
4. **Harder: Pinsker's inequality and the two distances.** `01/14` measured total variation $\mathrm{TV}(p, q) = \tfrac12\sum_x|p_x - q_x|$. Pinsker: $D(p\|q) \ge \frac{2}{\ln 2}\,\mathrm{TV}(p, q)^2$ (in bits). Check it on $20{,}000$ random pairs of four-outcome distributions; find the pairs that come closest to equality; and show the reverse bound is impossible by exhibiting a family with $\mathrm{TV}$ bounded (tending to $\tfrac12$) while $D \to \infty$ (hint: section 3(ii)). Then prove additivity (4a) for three independent factors by induction, and say why $\mathrm{TV}$ is *not* additive on products (compute $\mathrm{TV}(p^{\otimes 2}, q^{\otimes 2})$ against $2\,\mathrm{TV}(p, q)$).


## What you built

- You defined the relative entropy as cross-entropy minus entropy, computed it by hand term by term, and made `02/01`'s two overpayments exact: $\tfrac14$ bit each, $D(u\|p)$ and $D(p\|u)$.
- You proved Gibbs' inequality from $\ln x \le x - 1$ and spent it twice: $H \le \log_2 n$ (second proof, with the gap named) and the "$\ge$" of the source-coding theorem (every prefix code pays $D(p\|q) - \log_2 Z$ over the floor).
- You saw the three failures — asymmetry, $+\infty$ off the support, no triangle inequality — in two landscapes and three numbers, and why a coding cost has them.
- You derived additivity and the chain rule, checked joint convexity, and proved the data-processing inequality from the log-sum inequality: processing can only spend distinguishability.
- You named the three relative entropies ahead: mutual information, the Sinkhorn KL projection, and Umegaki's lift.

One inequality, $\ln x \le x - 1$, carried the whole notebook. The course will spend the quantity it controls in almost every notebook from here on.


**Dictionary so far** (module 01's table, `02/01`'s entropy row, plus today's row — the classical column is earned here; the quantum column is `02/09`'s promise, stated so you know where the thread goes):

| Classical | Quantum |
|-----------|---------|
| bit ($0$ or $1$) | qubit $a_0\,|0\rangle + a_1\,|1\rangle$ (two complex amplitudes) |
| probabilities only add (non-negative) | amplitudes add *and cancel* — interference; a relative phase $e^{i\varphi}$ is physical, a global phase is not |
| a bit's state space: two isolated points | a pure qubit's state space: the Bloch **sphere** (poles = the classical bit; equator = equal superpositions) |
| outcome probabilities $p_x$ | **Born rule** $p_x = \lvert a_x\rvert^2$ — the *shadow* one measurement casts of a state; a different measurement casts a different shadow |
| overlap of two probability vectors, $\sum_x \sqrt{p_x q_x}$ (Bhattacharyya coefficient) | **inner product** $\langle\varphi|\psi\rangle$ — the same sum with amplitudes in place of $\sqrt{p_x}$; its modulus squared is the yes-probability of "are you $|\varphi\rangle$?" (fidelity, `01/14`) |
| a reversible operation on a bit is a permutation (identity, NOT); a general operation on probability vectors is a stochastic matrix, usually lossy | a **gate** is a **unitary** matrix $U$ ($U^\dagger U = I$): preserves every inner product, is undone by $U^\dagger$, and rotates the Bloch sphere rigidly (half-angle in the matrix, full angle on the sphere); the general, lossy map is the quantum channel of `01/19` |
| a quantity is a function of the bit, $f(0), f(1)$; its average is $\sum_x p_x f(x)$ | an **observable** is a Hermitian matrix $A = A^\dagger$; its average is $\langle\psi|A|\psi\rangle$ (real), the outcomes are its eigenvalues; the Paulis are gates *and* observables, and $(\langle X\rangle,\langle Y\rangle,\langle Z\rangle)$ is the Bloch vector (measured shot by shot in `01/10`) |
| there is one way to read a bit: look at it | a **measurement basis** is any orthonormal pair $\{|b_0\rangle,|b_1\rangle\}$, $P(k) = |\langle b_k|\psi\rangle|^2$; the hardware natively reads only $Z$, so every other basis is reached by *rotate, then read* — $H$ for $X$, $S^\dagger$ then $H$ for $Y$ (the recombination and $Y$-type rules of `01/02`) |
| a bit has no continuous knob: you flip it or you do not | **rotation gates** $R_x, R_y, R_z$ (and phase gates $P(\lambda)$) are continuous dials — angle $\theta$ in the matrix as $\theta/2$, full $\theta$ on the sphere; any pure state is $R_z(\phi)R_y(\theta)|0\rangle$, any single-qubit gate is $e^{i\alpha}R_z R_y R_z$ (Nielsen & Chuang Thm 4.1) |
| the average $\sum_x p_x f(x)$ is estimated from $N$ samples with a standard error $\sqrt{\mathrm{Var}(f)/N}$ | the **expectation value** $\langle A\rangle = \langle\psi|A|\psi\rangle = \sum_k \lambda_k P(k)$ is the average of the eigenvalue outcomes; from $N$ shots of a $\pm1$ observable it is estimated with standard error $\sqrt{(1 - \langle A\rangle^2)/N} \le 1/\sqrt N$; three Pauli averages determine the state (tomography, `01/15`) |
| a randomized program on bits: initialize, apply logic gates in order, read the output; run it many times and estimate the output frequencies, with sampling error | a **circuit**: qubits initialized to $|0\rangle$, unitary gates in time order, measurement into classical bits; $N$ **shots** give **counts**, a sample of the Born shadow with the error bar above; the device runs a compiled **native** version ($R_z$, $\sqrt X$, $X$, one fixed entangler), equal to yours up to a global phase; a *systematic* miss that does not shrink with $N$ is the signature of noise, and needs $\rho$ (`01/12`) |
| probability vector $p = (p_0, p_1)$: non-negative entries summing to $1$; averages are $\sum_x p_x f(x)$; a fair coin is $(\tfrac12,\tfrac12)$ | **density matrix** $\rho = |\psi\rangle\langle\psi|$ (pure case): Hermitian, unit trace, positive; diagonal = the Born probabilities $p$, off-diagonal = the **coherence** $\tfrac12\sin\theta\,e^{-i\phi}$ (modulus: how much superposition; phase: the relative phase); $P(b) = \langle b|\rho|b\rangle$, $\langle A\rangle = \mathrm{tr}(\rho A)$, gates act as $U\rho U^\dagger$; $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ packs the Bloch vector; a global phase drops out; a $\rho$ that is diagonal in a shared basis *is* a classical $p$ — and the mixtures of `01/13` are the $\rho$'s that are no vector's outer product |
| not knowing which bit you hold is *the only* kind of uncertainty: a coin $p$; a fair coin $(\tfrac12,\tfrac12)$ is as uncertain as it gets, and a mixture of coins is a coin | a **mixed state** $\rho = \sum_i p_i|\psi_i\rangle\langle\psi_i|$ is the *second* kind of uncertainty (ignorance of which state), distinct from superposition (a definite state with random outcomes); its Bloch vector is the average of the parts' — inside the sphere, $I/2$ at the centre; **purity** $\mathrm{tr}(\rho^2) = \tfrac12(1 + |\vec r|^2) \in [\tfrac12, 1]$, eigenvalues $\tfrac12(1 \pm |\vec r|)$, von Neumann entropy = entropy of the eigenvalue coin (preview of `02/01`, `02/09`); a mixture forgets how it was prepared; **the seed:** $|+\rangle$ and $I/2$ share the diagonal $(\tfrac12,\tfrac12)$ and differ only in the corners |
| distances between probability vectors: **total variation** $\tfrac12\sum_x|p_x - q_x|$ (the best one-shot guess between $p$ and $q$ succeeds with probability $\tfrac12(1 + \mathrm{TV})$) and the **Bhattacharyya overlap** $\sum_x\sqrt{p_x q_x}$ | **trace distance** $T = \tfrac12\|\rho - \sigma\|_1 = \tfrac12|\vec r - \vec s|$ for qubits — half the distance in the Bloch **ball** (surface = pure, depth = mixedness); Helstrom: best one-shot guess succeeds with probability $\tfrac12(1 + T)$, by measuring the eigenbasis of $\rho - \sigma$; **fidelity** $F = (\mathrm{tr}\sqrt{\sqrt\rho\sigma\sqrt\rho})^2$ — *this course's convention*: $F = |\langle\varphi|\psi\rangle|^2 = \tfrac12(1 + \vec r\cdot\vec s)$ for pure states, $\langle\psi|\sigma|\psi\rangle$ for one pure, (Bhattacharyya)$^2$ for diagonal states; $F_U = \sqrt F$ is the Uhlmann fidelity of the Bures distance (`02/12`); $1 - \sqrt F \le T \le \sqrt{1 - F}$ |
| estimating a probability vector from data: count frequencies, quote $\pm\sqrt{p(1-p)/N}$; a miscalibrated counter biases every frequency by the same factor, and no number of samples fixes it — you calibrate the counter | **state tomography**: estimate $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ from three circuits' counts, each $\pm\sqrt{(1 - \langle\sigma\rangle^2)/N}$, and pack $\hat\rho = \tfrac12(I + \hat{\vec r}\cdot\vec\sigma)$ (project back onto the ball if noise pushes it out); purity, fidelity and trace distance read straight off $\hat\rho$; tomography reports the state *as seen through the meter* — a symmetric misassignment $\epsilon$ shrinks $\hat{\vec r}$ by $1 - 2\epsilon$, a lopsided one also shifts it (calibrate it away — exercise 2; `05/02` meets the same floor on a QPU) — while an idling qubit's *state* really does decohere, coherence as $e^{-t/T_2}$, population as $1 - e^{-t/T_1}$ (channels, `01/19`) |
| **marginal** of a joint table $p(a,b)$: sum over the other variable, $p_A(a) = \sum_b p(a,b)$ (row sums); independent variables factor, $p = p_A \otimes p_B$, and each marginal is the factor; *correlated* variables (the two matched coins) have perfectly random marginals although the pair is not random at all | two systems live on $\mathcal H_A \otimes \mathcal H_B$ (Kronecker product, index $k = 2a + b$, left factor first); a **product state** $\rho_A \otimes \rho_B$ is the independent case; the **partial trace** $\rho_A = \mathrm{tr}_B\,\rho_{AB}$, $(\rho_A)_{aa'} = \sum_b \rho_{ab,a'b}$ (replace each $2\times2$ block by its trace), is the only map that reproduces every local statistic $\mathrm{tr}(\rho_{AB}(M \otimes I)) = \mathrm{tr}(\rho_A M)$; a diagonal $\rho_{AB}$ reduces *exactly* to the row sums; a correlated joint state has **mixed** marginals — the *third* way a state can be mixed: it is part of something larger (`01/18` makes the whole pure and the parts still mixed) |
| **conditional logic** on two bits: controlled-NOT / XOR, $(a, b) \mapsto (a, a \oplus b)$ — a reversible *permutation* of the four outcomes; on a joint probability table it permutes the four entries; two independent operations, one per bit, can never make the bits agree when they did not before | **CNOT**, the $4\times4$ permutation unitary $|0\rangle\langle0|\otimes I + |1\rangle\langle1|\otimes X$ (controlled-$U$ pattern; control = left factor in this course, Qiskit's `cx(1, 0)`); it permutes the four basis vectors and, by linearity, acts on *superpositions* of the control: $\mathrm{CNOT}\,(\alpha|0\rangle + \beta|1\rangle)|0\rangle = \alpha|00\rangle + \beta|11\rangle$ — a product in, a non-product out (amplitude-matrix determinant $\ne 0$); no $M \otimes N$ can do that; CNOT$(H \otimes I)|00\rangle$ is the Bell state, whose corners are *coherence* between $|00\rangle$ and $|11\rangle$ (a product $|{+}{+}\rangle$ has them too; the coins have none) — entanglement proper is `01/18` |
| a joint table whose whole is *certain* (one outcome, probability $1$) has certain marginals; mixed marginals force a mixed whole (the coins); the correlation of two bits is fully described by their joint table, and any table is a mixture of point masses — every classical correlation is a *shared randomness* | **entanglement**: a *pure* joint state that is not a product (amplitude-matrix $\det\Psi \ne 0$) has **mixed parts** — pure whole, mixed parts, impossible for any table; measured by the **entanglement entropy** $S(\rho_A) = -\sum_i \lambda_i\log_2\lambda_i$ (eigenvalues of the part; $= S(\rho_B)$; $1$ bit for a Bell state; invariant under local gates $U\otimes V$) — **valid only when the whole is pure**: the coins have $S(\rho_A) = 1$ bit and no entanglement; a mixed state is **separable** if it is a mixture of products, and for two qubits the **partial transpose** decides it (a negative eigenvalue = entangled: Bell $-\tfrac12$, coins none); Bell and coins share every $Z$-statistic and differ in $X$: $\langle X\otimes X\rangle = 1$ vs $0$ |
| the general evolution of a probability vector is a **stochastic matrix** (Markov kernel) $p \to Tp$: columns are probability vectors, so probabilities stay non-negative and sum to $1$; a permutation is the reversible case; noise = a coin you did not see being tossed | a **quantum channel**: the general evolution $\rho \to \sum_k K_k\rho K_k^\dagger$ (Kraus operators, $\sum_k K_k^\dagger K_k = I$) — *derived* as "couple to an environment, evolve unitarily, trace the environment out" ($K_k = \langle k|_E\,U\,|0\rangle_E$); linear, trace-preserving and **completely** positive (the transpose is positive but sends half a Bell pair to a matrix with a $-\tfrac12$ eigenvalue — not a channel); a unitary is the reversible case ($K = U$), a stochastic matrix is the diagonal case ($K_{ij} = \sqrt{T_{ij}}\,|i\rangle\langle j|$); **decoherence = entanglement with an environment, then a partial trace**: dephasing (the environment learns the bit; coherence $\times e^{-t/T_\varphi}$), amplitude damping (energy leaks out; population $\times e^{-t/T_1}$, coherence $\times e^{-t/2T_1}$), depolarizing ($\rho \to (1-p)\rho + p\,I/2$); together they give $1/T_2 = 1/2T_1 + 1/T_\varphi$, hence $T_2 \le 2T_1$ — the laws `01/15` fitted |
| **Shannon entropy** $H(p) = -\sum_x p_x\log_2 p_x$: the average *surprise* $-\log_2 p_x$ (the log is forced by additivity of independent surprises); $0$ for a certain outcome, $\log_2 n$ for the uniform (Jensen), concave; **binary entropy** $h(p)$; joint and conditional entropy with the chain rule $H(AB) = H(A) + H(B|A)$, $H(B|A) \ge 0$, hence $H(A) \le H(AB)$ — a part is never more uncertain than the whole (coins: $1 = 1$; independent: $H(AB) = H(A) + H(B)$); the floor of lossless compression (Shannon 1948) | **von Neumann entropy** $S(\rho) = -\mathrm{tr}\,\rho\log_2\rho = H(\text{eigenvalues of }\rho)$ — the Shannon entropy of the eigenvalue coin (`01/13`): $0$ for a pure state, $1$ bit for $I/2$, $h\big(\tfrac{1 + |\vec r|}{2}\big)$ for a qubit; *but* the chain rule's positivity fails: the Bell state has $S(AB) = 0 < S(A) = 1$ (`01/18`) — a negative conditional entropy, `02/11`; the quantum side is earned in `02/09` |
| **relative entropy** (Kullback–Leibler) $D(p\|q) = \sum_x p_x\log_2\frac{p_x}{q_x}$ — the *extra* bits per symbol of coding with model $q$ when the truth is $p$ (cross-entropy minus entropy); **Gibbs**: $D \ge 0$, $= 0$ iff $p = q$ (from $\ln x \le x - 1$) — which re-proves $H \le \log_2 n$ and the source-coding floor; **not** a distance: asymmetric, no triangle inequality, $+\infty$ when $q$ gives zero to something $p$ can do; additive on products, jointly convex, and **monotone under any stochastic map** $D(Tp\|Tq) \le D(p\|q)$ — processing cannot create distinguishability; $D(p\,\|\,\text{uniform}) = \log_2 n - H(p)$ | **Umegaki relative entropy** $S(\rho\|\sigma) = \mathrm{tr}\,\rho(\log_2\rho - \log_2\sigma)$ (`02/09`): equals $D$ when $\rho, \sigma$ are diagonal in one basis, $+\infty$ when the support of $\rho$ leaves that of $\sigma$, $\ge 0$ (Klein), monotone under every quantum channel (Lindblad 1975) — and it *sees the corners*: $|{+}\rangle$ vs $I/2$ have identical diagonals, classical $D = 0$, quantum $S = 1$ bit |
| transport plan | quantum coupling |

## What's next

Relative entropy compares a model to the truth. Point it at a joint table and its own product of marginals and it answers a new question — how much do two variables share? That is the mutual information of `02/03_mutual_information`: non-negative by Gibbs, zero exactly when the table factorises, $1$ bit on the correlated coins of `01/16`, and the classical cousin of the quantum mutual information that reads the Bell state in `02/10`.


## References

- S. Kullback & R. A. Leibler, *On information and sufficiency*, Ann. Math. Statist. **22**, 79–86 (1951) — the definition, asymmetry acknowledged, the "extra information" reading. doi:10.1214/aoms/1177729694
- T. M. Cover & J. A. Thomas, *Elements of Information Theory*, 2nd ed., Wiley (2006) — §2.3 (relative entropy), Thm 2.6.3 (information inequality / Gibbs), Thm 2.7.1 (log-sum inequality, convexity), §2.8 (data-processing inequality), Lemma 11.6.1 (Pinsker), Thm 5.3.1 (§5.3: the source-coding "$\ge$" via $D$). doi:10.1002/047174882X
- C. E. Shannon, *A mathematical theory of communication*, Bell Syst. Tech. J. **27**, 379–423 (1948) — §9: the coding theorem whose converse is today's section 2. doi:10.1002/j.1538-7305.1948.tb01338.x
- I. Csiszár, *Information-type measures of difference of probability distributions and indirect observations*, Studia Sci. Math. Hungar. **2**, 299–318 (1967) — $D$ as the prototype $f$-divergence; monotonicity under Markov kernels.
- M. S. Pinsker, *Information and Information Stability of Random Variables and Processes*, Holden-Day (1964) — the inequality of exercise 4.
- H. Umegaki, *Conditional expectation in an operator algebra, IV (entropy and information)*, Kodai Math. Sem. Rep. **14**, 59–85 (1962) — the quantum relative entropy of `02/09`. doi:10.2996/kmj/1138844604
- G. Lindblad, *Completely positive maps and entropy inequalities*, Commun. Math. Phys. **40**, 147–151 (1975) — monotonicity of the Umegaki relative entropy under channels, the quantum 4(c). doi:10.1007/BF01609396
- I. Csiszár, *I-divergence geometry of probability distributions and minimization problems*, Ann. Probab. **3**, 146–158 (1975) — KL projections onto linear families, the Pythagorean identity, and the source of "Sinkhorn = KL projection" (`03/11`). doi:10.1214/aop/1176996454
- A. Uhlmann, *Relative entropy and the Wigner–Yanase–Dyson–Lieb concavity in an interpolation theory*, Commun. Math. Phys. **54**, 21–32 (1977) — monotonicity in full generality. doi:10.1007/BF01609834
- S. Amari & H. Nagaoka, *Methods of Information Geometry*, AMS/Oxford (2000) — KL projections and the Pythagorean relation behind `03/11`.

**Previous:** `notebooks/02_InformationAndGeometry/01_shannon_entropy.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/03_mutual_information.ipynb`
